---
description: Shared filesystem, configuration, and progress primitives for OCR providers
output-file: preprocessing.ocr.utils.html
title: Shared OCR utilities
---

In [ ]:
# | default_exp preprocessing.ocr.utils

In [ ]:
%load_ext autoreload
%autoreload 2

These provider-neutral helpers are shared by the GLM, DashScope, and Baidu Unlimited-OCR notebooks. Keeping them in one nbdev module prevents provider implementations from drifting apart.

In [ ]:
# | export
import asyncio
import os
from collections import Counter
from contextlib import AbstractAsyncContextManager
from dataclasses import dataclass
from functools import wraps
from pathlib import Path
from typing import (
    Any,
    Awaitable,
    Callable,
    Coroutine,
    Generic,
    ParamSpec,
    Protocol,
    Sequence,
    TypeVar,
    cast,
)

In [ ]:
# | export
_OCRFolderJob = TypeVar("_OCRFolderJob")
_OCRFolderResult = TypeVar("_OCRFolderResult", bound="OCRFolderResult")
_OCRFolderProgressResult = TypeVar(
    "_OCRFolderProgressResult", bound="OCRFolderResult", contravariant=True
)
_OCRFolderParams = ParamSpec("_OCRFolderParams")


class OCRFolderResult(Protocol):
    """Fields required from a provider-specific folder OCR result."""

    @property
    def source_path(self) -> Path: ...

    @property
    def status(self) -> str: ...

    @property
    def error(self) -> str | None: ...


class OCRFolderProgress(Protocol[_OCRFolderProgressResult]):
    """Progress renderer used by the shared folder workflow."""

    @property
    def notebook_mode(self) -> bool: ...

    def start_file(
        self, index: int, source_path: Path, pages_total: int | None = None
    ) -> None: ...

    def set_pages_total(self, index: int, pages_total: int) -> None: ...

    def complete_page(
        self,
        index: int,
        pages_completed: int,
        pages_total: int,
        elapsed_s: float,
    ) -> None: ...

    def finish_file(self, index: int) -> None: ...

    def record_result(
        self,
        index: int,
        result: _OCRFolderProgressResult,
        elapsed_s: float | None = None,
    ) -> None: ...

    def close(self) -> None: ...


@dataclass(frozen=True)
class OCRFolderExecution(Generic[_OCRFolderJob, _OCRFolderResult]):
    """Progress and worker callback for one prepared folder OCR run."""

    progress: OCRFolderProgress[_OCRFolderResult]
    run_job: Callable[
        [int, _OCRFolderJob],
        Coroutine[Any, Any, tuple[int, _OCRFolderResult, float]],
    ]


@dataclass(frozen=True)
class OCRFolderPlan(Generic[_OCRFolderJob, _OCRFolderResult]):
    """Provider hooks consumed by :func:`ocr_folder_workflow`."""

    operation_name: str
    job_count: int
    pending_jobs: Sequence[tuple[int, _OCRFolderJob]]
    initial_results: dict[int, _OCRFolderResult]
    execution: Callable[
        [],
        AbstractAsyncContextManager[OCRFolderExecution[_OCRFolderJob, _OCRFolderResult]],
    ]
    report_results: Callable[[Sequence[_OCRFolderResult]], None]


def _print_ocr_folder_summary(
    operation_name: str,
    event: str,
    results: Sequence[OCRFolderResult],
) -> None:
    counts = Counter(result.status for result in results)
    print(
        f"{operation_name} {event}: {counts['processed']} processed, "
        f"{counts['partial']} partial, {counts['skipped']} skipped, "
        f"{counts['failed']} failed"
    )


def ocr_folder_workflow(
    prepare: Callable[
        _OCRFolderParams,
        Awaitable[
            OCRFolderPlan[_OCRFolderJob, _OCRFolderResult]
            | list[_OCRFolderResult]
        ],
    ],
) -> Callable[_OCRFolderParams, Awaitable[list[_OCRFolderResult]]]:
    """Decorate provider setup with durable concurrent folder orchestration.

    The decorated coroutine performs provider setup and returns an
    :class:`OCRFolderPlan`. This wrapper owns concurrent task scheduling,
    deterministic result ordering, cancellation recovery, progress cleanup,
    summaries, reports, and provider-resource cleanup.
    """

    @wraps(prepare)
    async def wrapped(
        *args: _OCRFolderParams.args,
        **kwargs: _OCRFolderParams.kwargs,
    ) -> list[_OCRFolderResult]:
        plan = await prepare(*args, **kwargs)
        if isinstance(plan, list):
            return plan
        results_by_index = dict(plan.initial_results)
        async with plan.execution() as execution:
            progress = execution.progress
            tasks: list[
                asyncio.Task[tuple[int, _OCRFolderResult, float]]
            ] = [
                asyncio.create_task(execution.run_job(index, job))
                for index, job in plan.pending_jobs
            ]
            completion_records: list[str] = []
            cancelled_error: asyncio.CancelledError | None = None

            def remember_result(
                index: int, result: _OCRFolderResult, elapsed_s: float
            ) -> None:
                if index in results_by_index:
                    return
                results_by_index[index] = result
                progress.record_result(index, result, elapsed_s)
                if result.status != "skipped":
                    error_text = f" | error={result.error}" if result.error else ""
                    completion_records.append(
                        f"{plan.operation_name} task: {result.source_path} | "
                        f"status={result.status} | elapsed={elapsed_s:.2f}s"
                        f"{error_text}"
                    )

            try:
                for completed_task in asyncio.as_completed(tasks):
                    index, result, elapsed_s = await completed_task
                    remember_result(index, result, elapsed_s)
            except asyncio.CancelledError as error:
                cancelled_error = error
            finally:
                for task in tasks:
                    if not task.done():
                        task.cancel()
                await asyncio.gather(*tasks, return_exceptions=True)
                for task in tasks:
                    if task.cancelled():
                        continue
                    try:
                        index, result, elapsed_s = task.result()
                    except BaseException:
                        continue
                    remember_result(index, result, elapsed_s)
                progress.close()

            if not progress.notebook_mode:
                for record in completion_records:
                    print(record)

            if cancelled_error is not None:
                interrupted_results = [
                    results_by_index[index] for index in sorted(results_by_index)
                ]
                _print_ocr_folder_summary(
                    plan.operation_name, "interrupted", interrupted_results
                )
                if not progress.notebook_mode:
                    plan.report_results(interrupted_results)
                raise cancelled_error

            results = [
                results_by_index[index] for index in range(plan.job_count)
            ]
            _print_ocr_folder_summary(plan.operation_name, "complete", results)
            if not progress.notebook_mode:
                plan.report_results(results)
            return results

    return cast(
        Callable[_OCRFolderParams, Awaitable[list[_OCRFolderResult]]], wrapped
    )

In [ ]:
# | export
def find_project_root() -> Path:
    """Find the nearest parent containing ``pyproject.toml``."""
    starts: list[Path] = []
    module_file = globals().get("__file__")
    if isinstance(module_file, str):
        starts.append(Path(module_file).resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in [start, *start.parents]:
            if (candidate / "pyproject.toml").is_file():
                return candidate
    return Path.cwd().resolve()


def resolve_root(root_folder: Path | str) -> Path:
    """Resolve and validate an OCR input directory."""
    root = Path(root_folder).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"OCR root does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"OCR root is not a directory: {root}")
    return root


def resolve_output_dir_name(output_dir_name: str) -> str:
    """Validate one relative output-directory component."""
    normalized = output_dir_name.strip()
    path = Path(normalized)
    if (
        not normalized
        or path.is_absolute()
        or len(path.parts) != 1
        or normalized in {".", ".."}
    ):
        raise ValueError("output_dir_name must be one relative directory name")
    return normalized

In [ ]:
# | export
def positive_int(value: int | None, env_name: str, default: int) -> int:
    """Resolve a positive integer from an argument or environment variable."""
    if value is None:
        configured = os.getenv(env_name, "").strip()
        if configured:
            try:
                value = int(configured)
            except ValueError as error:
                raise ValueError(f"{env_name} must be a positive integer") from error
        else:
            value = default
    if value <= 0:
        raise ValueError(f"{env_name} must be a positive integer")
    return value


def value(obj: object, name: str, default: Any = None) -> Any:
    """Read a field from either a mapping or an attribute-based response."""
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)

In [ ]:
# | export
@dataclass
class ActivePageProgress:
    """Mutable page counters for one active OCR source."""

    source_path: Path
    pages_total: int | None = None
    pages_completed: int = 0
    last_page_elapsed_s: float | None = None


def running_in_notebook() -> bool:
    """Return whether output is being rendered by a Jupyter kernel."""
    try:
        from IPython import get_ipython
    except ImportError:
        return False
    shell = get_ipython()
    return shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell"

In [ ]:
# | hide
from contextlib import asynccontextmanager, redirect_stdout
from io import StringIO
from tempfile import TemporaryDirectory
from unittest.mock import patch

from fastcore.test import test_eq


def test_shared_ocr_utilities():
    assert (find_project_root() / "pyproject.toml").is_file()
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        test_eq(resolve_root(root), root.resolve())
        file_path = root / "file.txt"
        file_path.touch()
        with patch.dict(os.environ, {"OCR_TEST_CONCURRENCY": "3"}):
            test_eq(positive_int(None, "OCR_TEST_CONCURRENCY", 1), 3)
        test_eq(resolve_output_dir_name(".md"), ".md")
        test_eq(value({"status": "ok"}, "status"), "ok")
        state = ActivePageProgress(root, 2, 1, 0.25)
        test_eq((state.pages_total, state.pages_completed), (2, 1))


test_shared_ocr_utilities()

@dataclass
class _FakeFolderResult:
    source_path: Path
    status: str
    error: str | None = None


class _FakeFolderProgress:
    notebook_mode = False

    def __init__(self) -> None:
        self.recorded: list[int] = []
        self.closed = False

    def record_result(
        self, index: int, result: _FakeFolderResult, elapsed_s: float
    ) -> None:
        self.recorded.append(index)

    def close(self) -> None:
        self.closed = True


async def test_ocr_folder_workflow_decorator():
    progress = _FakeFolderProgress()
    cleaned = False
    reported: list[list[_FakeFolderResult]] = []
    skipped = _FakeFolderResult(Path("skipped.pdf"), "skipped")

    async def run_job(
        index: int, source_path: Path
    ) -> tuple[int, _FakeFolderResult, float]:
        await asyncio.sleep(0.01 if index == 1 else 0)
        return index, _FakeFolderResult(source_path, "processed"), 0.01

    @asynccontextmanager
    async def execution():
        nonlocal cleaned
        try:
            yield OCRFolderExecution(progress=progress, run_job=run_job)
        finally:
            cleaned = True

    @ocr_folder_workflow
    async def fake_folder() -> list[_FakeFolderResult]:
        return cast(
            Any,
            OCRFolderPlan(
                operation_name="Fake OCR",
                job_count=3,
                pending_jobs=[
                    (1, Path("slow.pdf")),
                    (2, Path("fast.pdf")),
                ],
                initial_results={0: skipped},
                execution=execution,
                report_results=lambda results: reported.append(list(results)),
            ),
        )

    with redirect_stdout(StringIO()) as output:
        results = await fake_folder()
    test_eq([result.source_path.name for result in results], [
        "skipped.pdf", "slow.pdf", "fast.pdf"
    ])
    test_eq(progress.recorded, [2, 1])
    assert progress.closed and cleaned
    test_eq(reported, [results])
    assert "Fake OCR complete: 2 processed" in output.getvalue()
    test_eq(fake_folder.__name__, "fake_folder")


await test_ocr_folder_workflow_decorator()